In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

Project root: C:\Users\Teddy\Documents\Code\rag_qasper


In [2]:
from rag_qasper.data import load_corpus

corpus, answer_qids = load_corpus()
dev_papers = corpus["dev"]
sample_paper = dev_papers[0]

print(f"Loaded {len(dev_papers)} dev papers.")
print(f"Sample paper ID: {sample_paper.paper_id}")
print(f"Title: {sample_paper.title}")
print(f"Paragraphs count: {len(sample_paper.paragraphs)}")
print(f"Questions count: {len(sample_paper.questions)}")

sample_q = sample_paper.questions[0]
print(f"\nExample Question: {sample_q.text}")
print(f"Gold paragraph IDs: {sample_q.gold_pids}")

c:\Users\Teddy\Documents\Code\rag_qasper\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 80 dev papers.
Sample paper ID: 1909.00326
Title: Towards Understanding Neural Machine Translation with Word Importance
Paragraphs count: 66
Questions count: 4

Example Question: Does their model suffer exhibit performance drops when incorporating word importance?
Gold paragraph IDs: []


In [3]:
from rag_qasper.chunking import add_headers, fixed_chunks, paragraph_chunks

fixed = fixed_chunks(sample_paper, size=800, overlap=200)
para = paragraph_chunks(sample_paper, max_chars=1100)
para_with_headers = add_headers(sample_paper, para)

print(f"Fixed chunks produced: {len(fixed)}")
print(f"Paragraph chunks produced: {len(para)}")
print("\n--- Example Paragraph Chunk text ---")
print(para[0]["text"][:200] + "...")
print("\n--- Example Indexed text with headers ---")
print(para_with_headers[0]["index_text"][:250] + "...")

Fixed chunks produced: 49
Paragraph chunks produced: 66

--- Example Paragraph Chunk text ---
Although neural machine translation (NMT) has advanced the state-of-the-art on various language pairs, the interpretability of NMT remains unsatisfactory. In this work, we propose to address this gap ...

--- Example Indexed text with headers ---
Title: Towards Understanding Neural Machine Translation with Word Importance
Section: Abstract
Although neural machine translation (NMT) has advanced the state-of-the-art on various language pairs, the interpretability of NMT remains unsatisfactory. ...


In [4]:
import numpy as np

from rag_qasper.embed import Embedder

embedder = Embedder(device="cpu", use_query_cache=True)

test_docs = ["Information retrieval evaluates recall and precision.", "Quantum computing uses qubits."]
doc_vecs = embedder.embed_documents(test_docs)
query_vec = embedder.embed_query("What does IR evaluate?")

print(f"Document embedding matrix shape: {doc_vecs.shape}")
print(f"Query vector dimension: {query_vec.shape}")
print(f"L2 Norm check (should be ~1.0): {np.linalg.norm(query_vec):.4f}")

sims = doc_vecs @ query_vec
print(f"Similarity with doc 0 (IR): {sims[0]:.4f}")
print(f"Similarity with doc 1 (Quantum): {sims[1]:.4f}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 366.76it/s]


Document embedding matrix shape: (2, 384)
Query vector dimension: (384,)
L2 Norm check (should be ~1.0): 1.0000
Similarity with doc 0 (IR): 0.5624
Similarity with doc 1 (Quantum): 0.4325


In [5]:
from rag_qasper.retriever.bm25 import BM25
from rag_qasper.retriever.dense import DenseRetriever
from rag_qasper.retriever.hybrid import HybridRetriever

chunks = para_with_headers
query = sample_q.text

bm25 = BM25([c.get("index_text", c["text"]) for c in chunks])
bm25_hits = bm25.search(query, top_k=3)

dense = DenseRetriever(chunks, embedder=embedder)
dense_hits = dense.search(query, top_k=3)

hybrid = HybridRetriever(chunks, embedder=embedder)
hybrid_hits = hybrid.search(query, top_k=3)

print("Query:", query)
print("\nTop BM25 hits (idx, score):", bm25_hits)
print("Top Dense hits (idx, score):", dense_hits)
print("Top Hybrid hits (idx, score):", hybrid_hits)

Query: Does their model suffer exhibit performance drops when incorporating word importance?

Top BM25 hits (idx, score): [(28, 6.832528973566857), (42, 5.108937763632686), (16, 4.012504211617251)]
Top Dense hits (idx, score): [(28, 0.7995828986167908), (58, 0.7660498023033142), (35, 0.758682906627655)]
Top Hybrid hits (idx, score): [(28, 0.03278688524590164), (35, 0.03057889822595705), (42, 0.029827662395050816)]


In [ ]:
import sys

from dotenv import load_dotenv

from scripts.generate import build_prompt, call_gemini, parse_response

# Remove cached modules from memory
for mod in list(sys.modules):
    if "rag_qasper" in mod or "scripts" in mod:
        del sys.modules[mod]

load_dotenv()

formatted_retrieved = [
    {"rank": i + 1, "text": chunks[idx]["text"]} 
    for i, (idx, _) in enumerate(hybrid_hits)
]

prompt = build_prompt(sample_q.text, formatted_retrieved)
print("--- Generated Prompt preview ---")
print(prompt[:400] + "...\n")

text, latency_ms = call_gemini(prompt)
answer, abstained = parse_response(text)


print(f"Model Raw Output: {text!r}")
print(f"Parsed Answer: {answer!r}")
print(f"Abstained: {abstained}")
print(f"Latency: {latency_ms:.1f} ms")

--- Generated Prompt preview ---

You answer a question about a scientific paper using ONLY the excerpts below.
Rules:
- Answer concisely: a short phrase or one sentence.
- For yes/no questions answer only "Yes" or "No".
- If the excerpts do not contain the answer, reply exactly: UNANSWERABLE

Excerpts:
[1] We evaluate the effectiveness of estimating word importance by the translation performance decrease. More specifically, unli...

Model Raw Output: 'Yes'
Parsed Answer: 'Yes'
Abstained: False
Latency: 16743.0 ms


In [6]:
import re

_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\[(])")

def sentence_ranges(text):
    out, prev = [], 0
    for m in _SENT_SPLIT.finditer(text):
        out.append((prev, m.start()))
        prev = m.end()
    out.append((prev, len(text)))
    return [(s, e) for s, e in out if e > s]  

Lanters_oath = "The brightest day the darkest night. No evil shall escape my sight? Let those who worship evil's might beware my power, Green Lantern's light!"
sentence_ranges(Lanters_oath)


[(0, 36), (37, 67), (68, 142)]

In [7]:
_TOKEN = re.compile(r"[a-z0-9]+")
_PLACEHOLDER = re.compile(r"^(bibref|tabref|figref|secref|floatref)\d*$")

def tokenize(text):
    return [t for t in _TOKEN.findall(text.lower()) if not _PLACEHOLDER.match(t)]
tokenize("This is a test sentence with bibref1 and figref2 placeholders.")

['this', 'is', 'a', 'test', 'sentence', 'with', 'and', 'placeholders']